In [ ]:
import dill
import json
from rich import print
from matplotlib import pyplot as plt
from matplotlib import rcParams as rc
from ipywidgets import interact, widgets
import numpy as np
import pandas as pd

rc["font.family"] = "Times New Roman"
rc["font.size"] = 14
rc["figure.figsize"] = (6, 4)
rc["axes.grid"] = True

In [ ]:
istart_channel:int = 3
iend_channel:int = 25
# Load the configuration file
conf = json.load(open("../../atem/data/atem.json"))
times = np.asarray(conf['channels'])[istart_channel:iend_channel] * 1e-6
n_turns = conf['n_turns']

In [ ]:
area:str = "NE"
path:str = f"../../atem/data/11-024_Alberta_{area}.csv"
dheader:list = [f"zoff30[{i}]" for i in range(istart_channel,iend_channel)]
picker:list = ["Line", "bheight", "TranPeak", "x_wgs84", "y_wgs84", "flight", 'pwrline'] + dheader # power line monitor

In [ ]:
raws = pd.read_csv(path)[picker]

In [ ]:
area:str = "NE"
path:str = f"../../atem/data/11-024_Alberta_{area}.csv"
dheader:list = [f"zoff30[{i}]" for i in range(istart_channel,iend_channel)]
picker:list = ["Line", "bheight", "TranPeak", "x_wgs84", "y_wgs84", "flight", 'pwrline'] + dheader # power line monitor

In [ ]:
raws = pd.read_csv(path)[picker]
xy = raws[["x_wgs84", "y_wgs84"]].to_numpy()
normalizer = (-1e-9)/ (raws["TranPeak"].values * n_turns).reshape(-1, 1)
raws[[f"zoff30[{i}]" for i in range(istart_channel, iend_channel)]] = raws[[f"zoff30[{i}]" for i in range(istart_channel, iend_channel)]] * normalizer

In [ ]:
line_no = list(raws["Line"].unique())

In [ ]:
nan_list = raws[raws["y_wgs84"].isna()]["Line"].unique()
print(nan_list)
counts = []
for line_n in nan_list:
    count = raws[raws["Line"] == line_n]["y_wgs84"].isna().sum()
    counts.append(count)
print(counts)

In [ ]:
plt.scatter(xy[:, 0], xy[:, 1], s=1, c='lightgray', label="All data")
i=0
for line_n in nan_list:
    test_x = raws[raws["Line"] == line_n]["x_wgs84"]
    test_y = raws[raws["Line"] == line_n]["y_wgs84"]
    plt.scatter(test_x, test_y, s=1, c=f"C{i}", label=line_n)
    i += 1
plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.title("Line having NaN values in y_wgs84")

In [ ]:
raws.dropna(subset=["y_wgs84"], inplace=True) # Remove rows with NaN values in y_wgs84
raws.fillna(1e-20, inplace=True) # Replace remaining NaN values with a small number (1e-20) to avoid issues in calculations

In [ ]:
dx = 50.
values = []
values_std = []
soundings = []
istart = 0
iend = len(line_no)

for i_line, line in enumerate(line_no[istart:iend]):
    df_line = raws[raws['Line']==line]

    # Calculate distance along the "Line"
    xy = df_line[["x_wgs84", "y_wgs84"]].to_numpy()
    distance = np.sqrt(((xy-xy[0,:])**2).sum(axis=1))
    # print(f"Number of NaN values in distance: {np.isnan(distance).sum()}")
    max_distance = distance.max()

    # Determine the no. of soundings per bin.
    if max_distance % dx ==0:
        n_sounding = int(max_distance / dx)
    else:
        n_sounding = int(np.round(max_distance / dx) + 1)

    # Create bins and assign each sounding to a bin
    bins = np.arange(n_sounding) * dx
    df_line.insert(0, 'distance', distance)
    # Bin distances
    df_line['bin'] = pd.cut(df_line['distance'], bins=bins)
    # Compute statistics per bin
    binned = (
        df_line.groupby('bin', observed=False)
            [['distance'] + picker[1:]]
            .mean()
    )
    binned.insert(0, 'Line', line)
    binned_std = (
        df_line.groupby('bin', observed=False)
            [['bheight'] + dheader]
            .std()
    )
    values.append(binned.values)
    values_std.append(binned_std.values)
    soundings.append(n_sounding)

df_data_binned = pd.DataFrame(data=np.vstack(values), columns=['Line', 'distance'] + picker[1:])
df_data_std_binned = pd.DataFrame(data=np.vstack(values_std), columns=['bheight'] + dheader)

In [ ]:
n_time = len(times[istart_channel:iend_channel])
dobs = df_data_binned[dheader].values.flatten() # Binned Data
del df_data_binned, df_data_std_binned

In [2]:
name:str = "./inv_results_atem_full.pik"
outDict:dict = dill.load(open(name, "rb"))
print(outDict.keys())

dict_keys([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15])

In [4]:
print(outDict[1])

{
    'iter': 1,
    'beta': 89442.61988428142,
    'phi_d': 396425523.0212637,
    'phi_m': 803.3366122511252,
    'f': 1504621616.6608224,
    'm': array([-2.45999371, -2.45823459, -2.45481963, ..., -2.4181495 ,
       -2.41716267, -2.41670127]),
    'dpred': array([-1.33317702e-09, -1.07304597e-09, -8.32068952e-10, ...,
       -3.74321817e-11, -3.09334073e-11, -2.54455332e-11]),
    'SparseSmallness.irls_threshold': 1e-08,
    'SparseSmallness.norm': 1,
    'r LaterallyConstrainedSmoothness.irls_threshold': 1e-08,
    'r LaterallyConstrainedSmoothness.norm': 1,
    'z LaterallyConstrainedSmoothness.irls_threshold': 1e-08,
    'z LaterallyConstrainedSmoothness.norm': 1
}

In [5]:
iterations = len(outDict.keys())
print(f"{iterations=}")

iterations=15

In [ ]:
# i_iteration = 0
def foo(i_iteration):
    DPRED = outDict[i_iteration]['DOBS'].reshape((n_sounding, n_time))
    DOBS = dobs.reshape((n_sounding, n_time))
    plt.plot(times, -DPRED[i_iteration,:])
    plt.yscale('log')
    plt.xscale('log')
    plt.title(f"x: {binned['x_wgs84'].values[i_iteration]:.1f}")
    plt.ylim(1e-12,1e-8)
interact(foo, i_iteration = widgets.IntSlider(min=0, max=n_sounding-1, continous_update=False))    
